# SAM3 CVAT Assist - chạy backend trên Google Colab

Dùng khi máy bạn không có GPU. Trước khi chạy: **Runtime > Change runtime type > GPU (T4 trở lên)**.

Notebook này chạy backend trên Colab và mở một đường hầm https tạm thời. Bạn dán địa chỉ đó vào tab **Cài đặt** của panel trên CVAT.

Lưu ý: ảnh frame sẽ đi từ trình duyệt của bạn tới máy Colab này. Chỉ dùng nếu dữ liệu của lớp được phép xử lý trên Colab.

## 1. Đưa repo lên Colab
Cách A: tải file `sam3-cvat-assist.zip` lên bằng ô dưới. Cách B: sửa ô thành `!git clone <địa chỉ repo của bạn>`.

In [ ]:
import os
from google.colab import files

if not os.path.exists('/content/sam3-cvat-assist'):
    uploaded = files.upload()  # chọn sam3-cvat-assist.zip
    !unzip -q -o sam3-cvat-assist.zip -d /content
%cd /content/sam3-cvat-assist
!ls

## 2. Cài thư viện
Colab có sẵn PyTorch. Ô này cài phần còn lại và SAM3 qua `transformers`.

In [ ]:
!pip -q install -r requirements.txt
!pip -q install --upgrade transformers huggingface_hub

## 3. Đăng nhập Hugging Face
Model `facebook/sam3` yêu cầu xin quyền: mở https://huggingface.co/facebook/sam3 , bấm đồng ý điều khoản, đợi được duyệt. Sau đó tạo access token (quyền Read) tại https://huggingface.co/settings/tokens và dán vào ô dưới.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Chạy backend + mở đường hầm
Ô này in ra **địa chỉ https** và **API key**. Giữ ô chạy suốt buổi làm việc. Mỗi lần chạy lại địa chỉ sẽ đổi.

In [ ]:
import os, re, secrets, subprocess, time, urllib.request

API_KEY = secrets.token_urlsafe(16)
env = dict(os.environ, SAM3_ENGINE='transformers', SAM3_HOST='127.0.0.1', SAM3_PORT='8765', SAM3_API_KEY=API_KEY)
server = subprocess.Popen(['python', '-m', 'backend'], env=env, stdout=open('backend.log', 'w'), stderr=subprocess.STDOUT)

print('Đang nạp SAM3 (lần đầu phải tải checkpoint, có thể mất vài phút)...')
for _ in range(600):
    if server.poll() is not None:
        print(open('backend.log').read()[-3000:])
        raise SystemExit('Backend dừng đột ngột. Xem log ở trên.')
    try:
        urllib.request.urlopen('http://127.0.0.1:8765/health', timeout=2)
        break
    except Exception:
        time.sleep(2)

if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8765'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if found:
        url = found.group(0)
        break
print('\nDán hai giá trị này vào tab "Cài đặt" của panel SAM3 Assist:')
print('  Địa chỉ backend:', url)
print('  API key        :', API_KEY)

In [ ]:
# Xem log backend khi cần
!tail -n 30 backend.log